# E3 — Entrenamiento PCN v5

**Cambio principal respecto a v4: fix de desalineación de centroide (H19)**

### El problema (H19)
Al generar una rotura eliminando 15-50% de puntos de un lado del objeto, el
centroide de la nube rota se desplaza hacia la región intacta (~0.4 unidades).
El GT permanece en el origen (0,0,0). El modelo tenía que aprender esta traslación
variable implícitamente → **colapsaba a placa plana** en los peores casos (CD≈0.18-0.23).

### El fix (en `dataset.py`)
```python
roto_mean = roto.mean(axis=0)
roto      = roto      - roto_mean   # centrar el fragmento en (0,0,0)
completo  = completo  - roto_mean   # desplazar GT al mismo frame
```
Ambas nubes quedan en el mismo frame de referencia. Es el estándar de FoldingNet y GRNet.

**Resumen de cambios v4 → v5:**
- **`CENTRAR_EN_ROTO=True`** en `dataset.py` ← cambio principal
- Todo lo demás igual a v4: datos v2, w_coarse=1.0, 500 épocas, batch=64

**Historial de métricas:**

| Versión | CD-L1 media | CD-L1 mediana | F-Score | Best epoch | Cambio clave |
|---------|-------------|---------------|---------|------------|-------------|
| v3      | 0.0665      | 0.0628        | 0.0243  | 347/400    | Baseline A100 |
| v4      | 0.0641      | 0.0570        | 0.0236  | 480/500    | Datos v2 + w_coarse=1.0 |
| **v5**  | ?           | ?             | ?       | ?          | **Fix centroide** |

⏱️ **Tiempo estimado en A100: ~2.5 horas**

---
### Antes de ejecutar:
Menú → **Entorno de ejecución → Cambiar tipo → A100 GPU**

In [2]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── CELDA 2: Clonar repo e instalar dependencias ───────────────
import os
from getpass import getpass

REPO_DIR = '/content/TFM'

if not os.path.exists(REPO_DIR):
    token = getpass('Pega tu token de GitHub (ghp_...) y pulsa Enter: ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    os.system(f'git clone {repo_url} {REPO_DIR}')
    del token
else:
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
os.system('git checkout raquel/e3')
print('Directorio de trabajo:', os.getcwd())

# Verificar que CENTRAR_EN_ROTO=True está activo en dataset.py
import subprocess
r = subprocess.run(['grep', 'CENTRAR_EN_ROTO', 'E3/dataset.py'],
                   capture_output=True, text=True)
print('\nVerificacion dataset.py:')
print(r.stdout.strip())

subprocess.run(['pip', 'install', 'torch', 'numpy', 'matplotlib', '--quiet'])
print('Dependencias instaladas.')

Pega tu token de GitHub (ghp_...) y pulsa Enter: ··········
Directorio de trabajo: /content/TFM

Verificacion dataset.py:

Dependencias instaladas.


In [4]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

VERSION = 'v5_pcn'

RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_v2'
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado'

# Referencias anteriores (para comparar al final)
RUTA_RESUMEN_V4 = f'{BASE_E3}/resultados/v4_pcn/resumen.txt'

RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print('Rutas v5:')
print(f'  sintetico_v2    : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  salida modelo   : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados: {RUTA_SALIDA_RESULTADOS}')

Rutas v5:
  sintetico_v2    : /content/drive/MyDrive/Datos_E2_E3/General/sintetico_roturas_v2
  fantastic_breaks: /content/drive/MyDrive/Datos_E2_E3/General/Fantastik_Break_Preprocesado
  salida modelo   : /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/modelos/v5_pcn
  salida resultados: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/resultados/v5_pcn


In [5]:
# ── CELDA 4: Verificar rutas ────────────────────────────────────
from pathlib import Path

rutas = {
    'sintetico_roturas_v2' : RUTA_SINTETICO,
    'fantastic_breaks'     : RUTA_FB_PROCESADO,
    'resumen v4 (ref)'     : RUTA_RESUMEN_V4,
}

for nombre, ruta in rutas.items():
    existe = Path(ruta).exists()
    print(f'  [{"OK" if existe else "FALTA"}] {nombre}: {ruta}')

n_sint = len(list(Path(RUTA_SINTETICO).glob('*_completo.npy'))) if Path(RUTA_SINTETICO).exists() else 0
n_fb   = len(list(Path(RUTA_FB_PROCESADO).glob('*_completo.npy'))) if Path(RUTA_FB_PROCESADO).exists() else 0
print(f'\nPares: sintetico_v2={n_sint}, fantastic_breaks={n_fb}, TOTAL={n_sint+n_fb}')

  [OK] sintetico_roturas_v2: /content/drive/MyDrive/Datos_E2_E3/General/sintetico_roturas_v2
  [OK] fantastic_breaks: /content/drive/MyDrive/Datos_E2_E3/General/Fantastik_Break_Preprocesado
  [OK] resumen v4 (ref): /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/resultados/v4_pcn/resumen.txt

Pares: sintetico_v2=2299, fantastic_breaks=61, TOTAL=2360


In [6]:
# ── CELDA 5: Copiar datos desde Drive ──────────────────────────
import subprocess
from pathlib import Path

def copiar_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.glob('*.npy')):
        n = len(list(dst.glob('*.npy')))
        print(f'  [OK] ya existe: {dst.name}  ({n} .npy)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    print(f'  Copiando {src.name}...', flush=True)
    r = subprocess.run(['rsync', '-a', '--no-links', f'{src}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR] {r.stderr[:300]}')
    else:
        print(f'  listo — {len(list(dst.glob("*.npy")))} .npy copiados.')

copiar_dir(RUTA_SINTETICO,    'Datos/sintetico/roturas_v2')
copiar_dir(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado')

print()
for c in ['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(c).glob('*.npy'))) if Path(c).exists() else 0
    print(f'  {c}: {n} .npy')

  Copiando sintetico_roturas_v2...
  listo — 4598 .npy copiados.
  Copiando Fantastik_Break_Preprocesado...
  listo — 122 .npy copiados.

  Datos/sintetico/roturas_v2: 4598 .npy
  Datos/fantastic_breaks/procesado: 122 .npy


In [7]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Sin GPU. Ve a Entorno de ejecucion → Cambiar tipo → A100 o T4')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [8]:
# ── CELDA 7: ENTRENAR v5 ───────────────────────────────────────
#
# Cambio respecto a v4: CENTRAR_EN_ROTO=True en dataset.py (fix H19)
# Todo lo demás igual: datos v2, w_coarse=1.0, 500 epocas, batch=64
#
# Con el fix de centroide, el modelo ya no tiene que aprender la traslacion
# variable entre roto y GT → se espera una reduccion notable de la varianza
# y mejora en los peores casos.

!python -m E3.train \
    --carpetas   Datos/sintetico/roturas_v2 Datos/fantastic_breaks/procesado \
    --epochs     500 \
    --lr         1e-4 \
    --lr_decay   100 \
    --batch_size 64 \
    --w_coarse   1.0

Dispositivo: cuda

=== Dataset E3 ===
  Total pares    : 2360
  Train          : 1888 (80%)
  Val            : 236   (10%)
  Test           : 236  (10%)
  Batch size     : 64
  Augmentación   : True
Parámetros entrenables: 5,157,379
Salida: coarse (512 pts) + fine (2048 pts)

  Época       Train         Val         LR   Tiempo
----------------------------------------------------
   1/500    0.407155    0.469335   1.00e-04     4.9s
  → best.pt guardado (val=0.469335)
   2/500    0.301173    0.290269   1.00e-04     3.2s
  → best.pt guardado (val=0.290269)
   3/500    0.277243    0.268690   1.00e-04     3.2s
  → best.pt guardado (val=0.268690)
   4/500    0.264533    0.255509   1.00e-04     3.2s
  → best.pt guardado (val=0.255509)
   5/500    0.252543    0.245333   1.00e-04     3.2s
  → best.pt guardado (val=0.245333)
   6/500    0.253710    0.246360   1.00e-04     3.2s
   7/500    0.238675    0.235397   1.00e-04     3.2s
  → best.pt guardado (val=0.235397)
   8/500    0.230575    0.22380

In [9]:
# ── CELDA 8: Guardar modelo en Drive ──────────────────────────
import shutil
from pathlib import Path

Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)
shutil.copy2('E3/checkpoints/best.pt', f'{RUTA_SALIDA_MODELO}/best.pt')
print(f'Modelo v5 guardado: {RUTA_SALIDA_MODELO}/best.pt')

for ckpt in sorted(Path('E3/checkpoints').glob('epoch_*.pt')):
    shutil.copy2(ckpt, Path(RUTA_SALIDA_MODELO) / ckpt.name)
    print(f'  + {ckpt.name}')

Modelo v5 guardado: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/modelos/v5_pcn/best.pt
  + epoch_005.pt
  + epoch_010.pt
  + epoch_015.pt
  + epoch_020.pt
  + epoch_025.pt
  + epoch_030.pt
  + epoch_035.pt
  + epoch_040.pt
  + epoch_045.pt
  + epoch_050.pt
  + epoch_055.pt
  + epoch_060.pt
  + epoch_065.pt
  + epoch_070.pt
  + epoch_075.pt
  + epoch_080.pt
  + epoch_085.pt
  + epoch_090.pt
  + epoch_095.pt
  + epoch_100.pt
  + epoch_105.pt
  + epoch_110.pt
  + epoch_115.pt
  + epoch_120.pt
  + epoch_125.pt
  + epoch_130.pt
  + epoch_135.pt
  + epoch_140.pt
  + epoch_145.pt
  + epoch_150.pt
  + epoch_155.pt
  + epoch_160.pt
  + epoch_165.pt
  + epoch_170.pt
  + epoch_175.pt
  + epoch_180.pt
  + epoch_185.pt
  + epoch_190.pt
  + epoch_195.pt
  + epoch_200.pt
  + epoch_205.pt
  + epoch_210.pt
  + epoch_215.pt
  + epoch_220.pt
  + epoch_225.pt
  + epoch_230.pt
  + epoch_235.pt
  + epoch_240.pt
  + epoch_245.pt
  + epoch_250.pt
  + epoch_255.pt
  + epoch_260.pt
  + epoch_265.pt
  + epoch_2

In [10]:
# ── CELDA 9: Evaluar modelo v5 ─────────────────────────────────
!python -m E3.evaluate \
    --checkpoint E3/checkpoints/best.pt \
    --carpetas   Datos/sintetico/roturas_v2 Datos/fantastic_breaks/procesado \
    --salida     E3/resultados

import shutil
from pathlib import Path

Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados v5 guardados: {RUTA_SALIDA_RESULTADOS}')

Dispositivo: cuda

=== Dataset E3 ===
  Total pares    : 2360
  Train          : 1888 (80%)
  Val            : 236   (10%)
  Test           : 236  (10%)
  Batch size     : 32
  Augmentación   : False
Muestras en test: 236
Checkpoint: E3/checkpoints/best.pt  (época 445)

=== Evaluación E3 — 236 muestras ===
Checkpoint : E3/checkpoints/best.pt  (época 445)
Nota       : val-loss del entrenamiento = CD(fine) + 0.5·CD(coarse).
             Aquí se reporta solo CD(fine) → número más bajo, es correcto.

Chamfer Distance (CD-L1):
  media    : 0.062954
  mediana  : 0.057170
  std      : 0.026518
  mejor    : 0.027497  (muestra 39)
  peor     : 0.222242  (muestra 22)

F-Score @ τ=0.01:
  media    : 0.0257
  mediana  : 0.0133
  std      : 0.0300
  mejor    : 0.1523  (muestra 186)
  peor     : 0.0000  (muestra 22)

CSV guardado: E3/resultados/metricas.csv

Generando 8 figuras (4 mejores + 4 peores)...
  figura_001_mejor_39.png
  figura_002_mejor_181.png
  figura_003_mejor_155.png
  figura_004_mejo

In [11]:
# ── CELDA 10: Comparar v3 / v4 / v5 ───────────────────────────
from pathlib import Path

versiones = [
    ('v3 (ref)',  None,                                          '0.066536', '0.062840', '0.0243', '347/400'),
    ('v4',        f'{BASE_E3}/resultados/v4_pcn/resumen.txt',   None,        None,       None,     None),
    ('v5',        'E3/resultados/resumen.txt',                  None,        None,       None,     None),
]

def leer_resumen(ruta):
    p = Path(ruta)
    if not p.exists():
        return {}
    datos = {}
    for line in p.read_text().splitlines():
        if ':' in line:
            k, v = line.split(':', 1)
            datos[k.strip()] = v.strip()
    return datos

print(f'{"Version":<10}  {"CD media":>10}  {"CD mediana":>10}  {"F-Score":>8}  {"Epoch"}')
print('-' * 58)

ref_cd = 0.066536
for nombre, ruta, cd_m, cd_med, fs, ep in versiones:
    if ruta is not None:
        d = leer_resumen(ruta)
        cd_m   = d.get('Chamfer Distance (CD-L1)', {}) # fallback
        # intentar leer campos del resumen.txt
        for line in Path(ruta).read_text().splitlines() if Path(ruta).exists() else []:
            if 'media' in line and ':' in line:  cd_m   = line.split(':')[1].strip()
            if 'mediana' in line and ':' in line: cd_med = line.split(':')[1].strip()
            if 'F-Score' in line and 'media' in line and ':' in line: fs = line.split(':')[1].strip()
            if 'epoca' in line.lower() and ':' in line: ep = line.split(':')[1].strip()

    try:
        mejora = f'{(ref_cd - float(cd_m)) / ref_cd * 100:+.1f}%' if nombre != 'v3 (ref)' else '—'
    except Exception:
        mejora = '?'

    print(f'{nombre:<10}  {str(cd_m):>10}  {str(cd_med):>10}  {str(fs):>8}  {ep}  {mejora}')

print('\nNota: CD-L1 menor = mejor | F-Score mayor = mejor')

Version       CD media  CD mediana   F-Score  Epoch
----------------------------------------------------------
v3 (ref)      0.066536    0.062840    0.0243  347/400  —
v4              0.0123      0.0123      None  None  +81.5%
v5              0.0133      0.0133      None  None  +80.0%

Nota: CD-L1 menor = mejor | F-Score mayor = mejor


In [12]:
# ── CELDA 11: Visualizacion 3D interactiva ─────────────────────
#   Azul  = entrada rota (ya centrada en su centroide)
#   Verde = GT completa (en el mismo frame que la rota)
#   Rojo  = prediccion del modelo

import subprocess
subprocess.run(['pip', 'install', 'plotly', '--quiet'])

import plotly.graph_objects as go
import numpy as np
import torch
from E3.train import PCN, chamfer_distance
from E3.dataset import construir_dataloaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PCN().to(device)
ckpt   = torch.load('E3/checkpoints/best.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Modelo v5 cargado — epoca {ckpt["epoch"]}')

_, _, test_loader = construir_dataloaders(
    carpetas=['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado'],
    batch_size=32, augmentar=False,
)

rotos, gts, preds, cds = [], [], [], []
with torch.no_grad():
    for roto_b, gt_b in test_loader:
        _, pred_b = model(roto_b.to(device))
        pred_b = pred_b.cpu()
        for i in range(len(roto_b)):
            p, g = pred_b[i:i+1], gt_b[i:i+1]
            cd = chamfer_distance(p, g).item()
            rotos.append(roto_b[i].numpy())
            gts.append(gt_b[i].numpy())
            preds.append(pred_b[i].numpy())
            cds.append(cd)

cds_arr = np.array(cds)
orden   = np.argsort(cds_arr)
indices = list(orden[:3]) + list(orden[-3:])
titulos = ['Mejor 1', 'Mejor 2', 'Mejor 3', 'Peor 1', 'Peor 2', 'Peor 3']
print(f'CD — mejor: {cds_arr[orden[0]]:.4f} | peor: {cds_arr[orden[-1]]:.4f} | media: {cds_arr.mean():.4f}')
print(f'std: {cds_arr.std():.4f}  (v4 fue 0.0283 — deberia bajar con el fix de centroide)')

for idx, titulo in zip(indices, titulos):
    fig = go.Figure([
        go.Scatter3d(x=rotos[idx][:,0], y=rotos[idx][:,1], z=rotos[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#4C72B0', opacity=0.55), name='Rota'),
        go.Scatter3d(x=gts[idx][:,0],   y=gts[idx][:,1],   z=gts[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#55A868', opacity=0.35), name='GT'),
        go.Scatter3d(x=preds[idx][:,0], y=preds[idx][:,1], z=preds[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#C44E52', opacity=0.85), name='Pred'),
    ])
    fig.update_layout(
        title=f'{titulo} — CD={cds[idx]:.4f}',
        scene=dict(xaxis=dict(range=[-1,1], showticklabels=False),
                   yaxis=dict(range=[-1,1], showticklabels=False),
                   zaxis=dict(range=[-1,1], showticklabels=False),
                   aspectmode='cube'),
        height=500, margin=dict(l=0,r=0,b=0,t=40)
    )
    fig.show()

Modelo v5 cargado — epoca 445

=== Dataset E3 ===
  Total pares    : 2360
  Train          : 1888 (80%)
  Val            : 236   (10%)
  Test           : 236  (10%)
  Batch size     : 32
  Augmentación   : False
CD — mejor: 0.0275 | peor: 0.2222 | media: 0.0630
std: 0.0265  (v4 fue 0.0283 — deberia bajar con el fix de centroide)
